# Synthetic data for MS MARCO

This notebook builds a contextual faithfulness dataset from **MS MARCO validation set**.

Pipeline:
1. Load MS MARCO validation set from Hugging Face using `HF_TOKEN` from Kaggle Secrets.
2. Parse the dataset into `id`, `question`, `context`, `gold_answer`.
3. Keep the 500 samples with the longest answers.
4. Use OpenAI API to generate one hallucinated answer per sample.
5. Export:
   - `msmarcro_500.csv`
   - `hallu_msmacro.csv`
   - `labeled_msmacro.csv`

Task framing: the negative answer should be **not supported by the provided context**, not merely stylistically different.


In [1]:
import os
import time
import json
import re
from typing import Any, Dict, List, Optional

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset
from openai import OpenAI

import dotenv
dotenv.load_dotenv()


c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 1. Configuration

In [ ]:
# ==============================
# Dataset config
# ==============================

# Current Hugging Face MS MARCO dataset.
# If your environment has another mirror, change DATASET_NAME / CONFIG_NAME here.
DATASET_NAME = "microsoft/ms_marco"
CONFIG_NAME = "v1.1"
SPLIT = "validation"

N_SAMPLES = 500

# Context parsing
SELECTED_ONLY = True      # Prefer gold/selected passages when available
MAX_PASSAGES = 10         # Safety cap if fallback to all passages
MIN_ANSWER_WORDS = 5      # Remove empty/noisy answers
MIN_CONTEXT_WORDS = 20    # Remove samples with unusable context

# Generation config
MODEL_NAME = "gpt-4o-mini"
SLEEP_TIME = 1
CHECKPOINT_PATH = "../results/synthetic-data/msmacro_checkpoint.csv"

# Output paths
MSMARCO_500_PATH = "../data/ms-macro/msmarco_500.csv"       # keep user's requested filename
HALLU_PATH = "../data/ms-macro/hallu_msmacro.csv"
LABELED_PATH = "../data/ms-macro/labeled_msmacro.csv"


In [3]:
if not os.path.exists("../data/ms-macro"):
    os.makedirs("../data/ms-macro")

## 2. Load secrets and clients

In [4]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Add OPENAI_API_KEY to Kaggle Secrets or environment variables.")

client = OpenAI(api_key=OPENAI_API_KEY)


## 3. Load MS MARCO validation set from Hugging Face

In [5]:
def load_ms_marco_dataset():
    """Load MS MARCO validation set.

    Newer versions of `datasets` use `token=...`.
    Older versions may still use `use_auth_token=...`.
    """

    ds = load_dataset("microsoft/ms_marco", "v1.1")
    ds = ds['validation']
    return ds

raw_ds = load_ms_marco_dataset()
print(raw_ds)
print(raw_ds.column_names)


c:\Users\vnpq2\anaconda3\envs\ragas-env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vnpq2\.cache\huggingface\hub\datasets--microsoft--ms_marco. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


KeyboardInterrupt: 

In [5]:
# Quick inspection
raw_ds[0]


{'answers': ['Approximately $15,000 per year.'],
 'passages': {'is_selected': [1, 0, 0, 0, 0, 0],
  'passage_text': ['The average Walgreens salary ranges from approximately $15,000 per year for Customer Service Associate / Cashier to $179,900 per year for District Manager. Average Walgreens hourly pay ranges from approximately $7.35 per hour for Laboratory Technician to $68.90 per hour for Pharmacy Manager. Salary information comes from 7,810 data points collected directly from employees, users, and jobs on Indeed.',
   'The average revenue in 2011 of a Starbuck Store was $1,078,000, up  from $1,011,000 in 2010.    The average ticket (total purchase) at domestic Starbuck stores in  No … vember 2007 was reported at $6.36.    In 2008, the average ticket was flat (0.0% change).',
   'In fiscal 2014, Walgreens opened a total of 184 new locations and acquired 84 locations, for a net decrease of 273 after relocations and closings. How big are your stores? The average size for a typical Walgr

## 4. Parsing helpers

In [6]:
def normalize_text(x: Any) -> str:
    """Convert MS MARCO field values into clean plain text."""
    if x is None:
        return ""

    if isinstance(x, str):
        return re.sub(r"\s+", " ", x).strip()

    if isinstance(x, (list, tuple, np.ndarray)):
        parts = [normalize_text(v) for v in x]
        parts = [p for p in parts if p and p.lower() not in {"no answer present.", "no answer present"}]
        return " ".join(parts).strip()

    return re.sub(r"\s+", " ", str(x)).strip()


def pick_gold_answer(row: Dict[str, Any]) -> str:
    """Pick the longest available answer for a sample.

    MS MARCO usually stores answers as a list in `answers`.
    We choose the longest non-empty answer because this notebook focuses on long-form samples.
    """
    answers = row.get("answers", row.get("answer", ""))

    if isinstance(answers, (list, tuple, np.ndarray)):
        candidates = [
            normalize_text(a)
            for a in answers
            if normalize_text(a)
            and normalize_text(a).lower() not in {"no answer present.", "no answer present"}
        ]
        if not candidates:
            return ""
        return max(candidates, key=lambda s: len(s.split()))

    answer = normalize_text(answers)
    if answer.lower() in {"no answer present.", "no answer present"}:
        return ""
    return answer


def _get_passage_list(passages: Any) -> List[Dict[str, Any]]:
    """Normalize MS MARCO passages into a list of dicts.

    Expected MS MARCO HF structure is usually:
    {
        "is_selected": [...],
        "passage_text": [...],
        "url": [...]
    }

    This function also handles already-list-like formats defensively.
    """
    if passages is None:
        return []

    # HF common case: dict of lists
    if isinstance(passages, dict):
        texts = passages.get("passage_text", passages.get("text", []))
        selected = passages.get("is_selected", [None] * len(texts))
        urls = passages.get("url", [""] * len(texts))

        result = []
        for i, text in enumerate(texts):
            result.append({
                "passage_text": normalize_text(text),
                "is_selected": selected[i] if i < len(selected) else None,
                "url": urls[i] if i < len(urls) else "",
            })
        return result

    # Defensive case: list of dicts or list of strings
    if isinstance(passages, list):
        result = []
        for p in passages:
            if isinstance(p, dict):
                result.append({
                    "passage_text": normalize_text(p.get("passage_text", p.get("text", ""))),
                    "is_selected": p.get("is_selected", p.get("selected", None)),
                    "url": p.get("url", ""),
                })
            else:
                result.append({
                    "passage_text": normalize_text(p),
                    "is_selected": None,
                    "url": "",
                })
        return result

    return [{"passage_text": normalize_text(passages), "is_selected": None, "url": ""}]


def parse_context(passages: Any, selected_only: bool = True, max_passages: int = 10) -> str:
    """Build a clean gold context string.

    Design:
    1. Prefer passages marked `is_selected == 1` because they are closer to gold context.
    2. If no selected passage exists, fallback to the first `max_passages` non-empty passages.
    3. Keep passage boundaries with [P1], [P2], ... so the model can reason over evidence.
    """
    passage_list = _get_passage_list(passages)
    passage_list = [p for p in passage_list if p["passage_text"]]

    if selected_only:
        selected_passages = [
            p for p in passage_list
            if str(p.get("is_selected", "")).lower() in {"1", "true", "yes"}
        ]
    else:
        selected_passages = []

    chosen = selected_passages if selected_passages else passage_list[:max_passages]

    context_blocks = []
    for i, p in enumerate(chosen, start=1):
        url = normalize_text(p.get("url", ""))
        text = p["passage_text"]

        if url:
            context_blocks.append(f"[P{i}] Source: {url}\\n{text} ")
        else:
            context_blocks.append(f"[P{i}] {text}")

    return "\\n\\n".join(context_blocks).strip()


## 5. Convert MS MARCO into `id`, `question`, `context`, `gold_answer`

In [7]:
def convert_ms_marco_to_rag_df(dataset) -> pd.DataFrame:
    records = []

    for idx, row in enumerate(tqdm(dataset, desc="Parsing MS MARCO")):
        row = dict(row)

        sample_id = normalize_text(
            row.get("query_id", row.get("id", idx))
        )
        if not sample_id:
            sample_id = str(idx)

        question = normalize_text(
            row.get("query", row.get("question", ""))
        )

        gold_answer = pick_gold_answer(row)
        context = parse_context(
            row.get("passages", row.get("context", "")),
            selected_only=SELECTED_ONLY,
            max_passages=MAX_PASSAGES,
        )

        records.append({
            "id": str(sample_id),
            "question": question,
            "context": context,
            "gold_answer": gold_answer,
        })

    df = pd.DataFrame(records)

    df["answer_len"] = df["gold_answer"].fillna("").str.split().str.len()
    df["context_len"] = df["context"].fillna("").str.split().str.len()

    df = df[
        df["question"].notnull() &
        df["context"].notnull() &
        df["gold_answer"].notnull() &
        (df["question"].str.len() > 0) &
        (df["context_len"] >= MIN_CONTEXT_WORDS) &
        (df["answer_len"] >= MIN_ANSWER_WORDS)
    ].copy()

    return df


msmarco_df = convert_ms_marco_to_rag_df(raw_ds)
print(msmarco_df.shape)
msmarco_df.head()


Parsing MS MARCO:   0%|          | 0/10047 [00:00<?, ?it/s]

(6089, 6)


,id,question,context,gold_answer,answer_len,context_len
1,9653,how much do bartenders make,[P1] Source: http://www.breakintobartending.co...,The average hourly wage for a bartender is $10...,16,78
2,9654,what is a furuncle boil,[P1] Source: https://en.wikipedia.org/wiki/Boi...,"A boil, also called a furuncle, is a deep foll...",26,99
3,9655,what can urinalysis detect,[P1] Source: http://www.mayoclinic.org/tests-p...,"Detect and assess a wide range of disorders, s...",17,81
4,9656,what is vitamin a used for,[P1] Source: http://www.webmd.com/vitamins-sup...,"Shigellosis, diseases of the nervous system, n...",32,77
5,9657,what causes genetic alterations in normal cells,[P1] Source: http://www.hindawi.com/journals/i...,The initiation of cell transformation is gener...,26,70


In [8]:
print(msmarco_df.iloc[0]['gold_answer'])

The average hourly wage for a bartender is $10.36 and the average yearly take-home is $21,550.


In [9]:
# Pick 500 samples with the longest gold answers
msmarco_500 = (
    msmarco_df
    .sort_values(["answer_len", "context_len"], ascending=False)
    .head(N_SAMPLES)
    .reset_index(drop=True)
)

# Keep only required columns in the exported clean dataset
msmarco_500_export = msmarco_500[["id", "question", "context", "gold_answer"]].copy()
msmarco_500_export.to_csv(MSMARCO_500_PATH, index=False, encoding="utf-8-sig")

print(msmarco_500_export.shape)
print(f"Saved: {MSMARCO_500_PATH}")
msmarco_500_export.head()


(500, 4)
Saved: /kaggle/working/msmarcro_500.csv


,id,question,context,gold_answer
0,10736,buying a car dealer vs. private seller,[P1] Source: http://www.consumerhelp.ie/buying...,Traders are often called “dealers” and sell ca...
1,12719,convert dollar pay into salary to twice a month,[P1] Source: http://www.ehow.com/how_12044390_...,Determine what your paydays will be throughout...
2,17806,how to plant dwarf apple trees,[P1] Source: http://www.gardenguides.com/79766...,Locate a site for the dwarf apple tree where i...
3,14675,what arthritis is,[P1] Source: http://www.healthline.com/health/...,Arthritis is inflammation of the joints (the p...
4,16014,how to record audio with iphone,[P1] Source: http://www.solveyourtech.com/reco...,1: Open the Voice Memos app. If you cannot fin...


## 6. Prompt for hallucinated answer generation

In [10]:
PROMPT_SYSTEM = """You are an AI that generates hallucinated answers for evaluating contextual faithfulness.

Follow all instructions strictly:
- The answer MUST NOT be fully supported by the provided context.
- The answer should directly conflict with, exaggerate, or add 1-2 unsupported key claims relative to the context.
- The answer should remain fluent, natural, and confident.
- Keep the style and length similar to the gold answer.
- Do NOT mention uncertainty.
- Do NOT say the answer is hallucinated, unsupported, false, or incorrect.
- Do NOT use special formatting, markdown, or bullet points.
- Output plain text only.

The goal is contextual unfaithfulness, not random nonsense.
"""

PROMPT_USER = """Question:
{question}

Gold Answer:
{gold_answer}

Context:
{context}

Return ONLY the hallucinated answer.
"""


In [11]:
def synthetic(row: pd.Series, client: OpenAI, model: str = MODEL_NAME) -> Optional[str]:
    prompt_user = PROMPT_USER.format(
        question=str(row["question"]).strip(),
        gold_answer=str(row["gold_answer"]).strip(),
        context=str(row["context"]).strip(),
    )

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": PROMPT_SYSTEM},
                {"role": "user", "content": prompt_user},
            ],
            temperature=0.7,
        )

        text = response.choices[0].message.content.strip()

        # Basic validation
        if not text:
            return None

        if text.strip().lower() == str(row["gold_answer"]).strip().lower():
            return None

        return text

    except Exception as e:
        print(f"OpenAI error for id={row.get('id', 'unknown')}: {e}")
        return None


## 7. Test generation on one sample

In [12]:
row = msmarco_500_export.iloc[0]

hallu = synthetic(
    row=row,
    client=client,
    model=MODEL_NAME,
)

print("=== ID ===")
print(row["id"])

print("\n=== QUESTION ===")
print(row["question"])

print("\n=== CONTEXT ===")
print(row["context"][:1500])

print("\n=== GOLD ANSWER ===")
print(row["gold_answer"])

print("\n=== HALLUCINATED ANSWER ===")
print(hallu)


=== ID ===
10736

=== QUESTION ===
buying a car dealer vs. private seller

=== CONTEXT ===
[P1] Source: http://www.consumerhelp.ie/buying-car-dealer-private\nBuying a car-dealer or private seller. There are two main sources of buying a second hand car: traders and private sellers. Traders are often called “dealers” and sell cars as part of their business. Private sellers generally have only one car to sell and are not selling it as part of their business. Your consumer rights vary depending on whether you buy from a dealer or private seller. If you buy from a dealer, you have some protection under consumer law. However, if you buy privately, you do not have the same consumer rights because the person selling the car is not acting as a business.

=== GOLD ANSWER ===
Traders are often called “dealers” and sell cars as part of their business.Private sellers generally have only one car to sell and are not selling it as part of their business.Your consumer rights vary depending on whether y

## 8. Batch generation with checkpoint resume

In [13]:
def load_checkpoint(checkpoint_path: str) -> pd.DataFrame:
    if checkpoint_path and os.path.exists(checkpoint_path):
        return pd.read_csv(checkpoint_path, dtype={"id": str})
    return pd.DataFrame(columns=["id", "hallu_answer"])


def save_checkpoint(rows: List[Dict[str, str]], checkpoint_path: str) -> None:
    ckpt = pd.DataFrame(rows)
    ckpt.to_csv(checkpoint_path, index=False, encoding="utf-8-sig")


def synthetic_batch(
    df: pd.DataFrame,
    client: OpenAI,
    model: str = MODEL_NAME,
    checkpoint_path: Optional[str] = CHECKPOINT_PATH,
    checkpoint_freq: int = 25,
) -> pd.DataFrame:
    df = df.copy()
    df["id"] = df["id"].astype(str)

    checkpoint_df = load_checkpoint(checkpoint_path) if checkpoint_path else pd.DataFrame(columns=["id", "hallu_answer"])
    done_ids = set(checkpoint_df["id"].astype(str).tolist())

    rows = checkpoint_df.to_dict("records")

    pbar = tqdm(total=len(df), initial=len(done_ids), desc="Generating hallucinations", unit="sample")

    for _, row in df.iterrows():
        sample_id = str(row["id"])

        if sample_id in done_ids:
            continue

        hallucinated = synthetic(row=row, client=client, model=model)
        if hallucinated is None:
            hallucinated = ""

        rows.append({
            "id": sample_id,
            "hallu_answer": hallucinated,
        })
        done_ids.add(sample_id)

        pbar.update(1)

        if checkpoint_path and len(rows) % checkpoint_freq == 0:
            save_checkpoint(rows, checkpoint_path)
            print(f"💾 checkpoint saved: {len(rows)}/{len(df)}")

        time.sleep(SLEEP_TIME)

    pbar.close()

    if checkpoint_path:
        save_checkpoint(rows, checkpoint_path)
        print(f"✅ final checkpoint saved: {checkpoint_path}")

    return pd.DataFrame(rows)


## 9. Build hallucinated and labeled datasets

In [14]:
def build_hallu_dataset(
    base_df: pd.DataFrame,
    client: OpenAI,
    model: str = MODEL_NAME,
    checkpoint_path: str = CHECKPOINT_PATH,
    hallu_path: str = HALLU_PATH,
    labeled_path: str = LABELED_PATH,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    base_df = base_df.copy()
    base_df["id"] = base_df["id"].astype(str)

    # 1. Generate hallucinated answers
    ckpt = synthetic_batch(
        base_df,
        client=client,
        model=model,
        checkpoint_path=checkpoint_path,
        checkpoint_freq=25,
    )

    ckpt["id"] = ckpt["id"].astype(str)

    # 2. Merge back with original samples
    hallu_df = base_df.merge(ckpt, on="id", how="left")

    # 3. Filter empty generations
    empty_mask = hallu_df["hallu_answer"].fillna("").str.len() < 5
    n_empty = int(empty_mask.sum())
    print(f"Empty/invalid hallucinations: {n_empty}")

    if n_empty > 0:
        print("Retrying empty/invalid generations once...")
        retry_df = hallu_df.loc[empty_mask, base_df.columns].copy()
        retry_rows = []

        for _, row in tqdm(retry_df.iterrows(), total=len(retry_df), desc="Retrying"):
            hallucinated = synthetic(row=row, client=client, model=model)
            retry_rows.append({
                "id": str(row["id"]),
                "hallu_answer": hallucinated or "",
            })
            time.sleep(SLEEP_TIME)

        retry_ckpt = pd.DataFrame(retry_rows)
        retry_map = dict(zip(retry_ckpt["id"].astype(str), retry_ckpt["hallu_answer"]))

        hallu_df.loc[empty_mask, "hallu_answer"] = hallu_df.loc[empty_mask, "id"].map(retry_map)

    hallu_df = hallu_df[
        hallu_df["hallu_answer"].notnull() &
        (hallu_df["hallu_answer"].str.len() >= 5)
    ].copy()

    # 4. Export hallu dataset with same schema as gold dataset + answer field
    hallu_export = hallu_df[["id", "question", "context", "gold_answer", "hallu_answer"]].copy()
    hallu_export.to_csv(hallu_path, index=False, encoding="utf-8-sig")
    print(f"Saved hallucinated dataset: {hallu_path}")

    # 5. Build labeled dataset
    # label = 1: supported/gold answer
    # label = 0: hallucinated/unsupported answer
    positive = base_df[["id", "question", "context", "gold_answer"]].copy()
    positive = positive.rename(columns={"gold_answer": "answer"})
    positive["label"] = 1

    negative = hallu_export[["id", "question", "context", "hallu_answer"]].copy()
    negative = negative.rename(columns={"hallu_answer": "answer"})
    negative["id"] = negative["id"].astype(str) + "_hallu"
    negative["label"] = 0

    labeled = pd.concat([positive, negative], ignore_index=True)
    labeled.to_csv(labeled_path, index=False, encoding="utf-8-sig")
    print(f"Saved labeled dataset: {labeled_path}")
    print(labeled["label"].value_counts())

    return labeled, hallu_export


In [15]:
labeled_msmacro, hallu_msmacro = build_hallu_dataset(
    msmarco_500_export,
    client=client,
    model=MODEL_NAME,
    checkpoint_path=CHECKPOINT_PATH,
    hallu_path=HALLU_PATH,
    labeled_path=LABELED_PATH,
)

labeled_msmacro.head()


Generating hallucinations:   0%|          | 0/500 [00:00<?, ?sample/s]

💾 checkpoint saved: 25/500
💾 checkpoint saved: 50/500
💾 checkpoint saved: 75/500
💾 checkpoint saved: 100/500
💾 checkpoint saved: 125/500
💾 checkpoint saved: 150/500
💾 checkpoint saved: 175/500
💾 checkpoint saved: 200/500
💾 checkpoint saved: 225/500
💾 checkpoint saved: 250/500
💾 checkpoint saved: 275/500
💾 checkpoint saved: 300/500
💾 checkpoint saved: 325/500
💾 checkpoint saved: 350/500
💾 checkpoint saved: 375/500
💾 checkpoint saved: 400/500
💾 checkpoint saved: 425/500
💾 checkpoint saved: 450/500
💾 checkpoint saved: 475/500
💾 checkpoint saved: 500/500
✅ final checkpoint saved: /kaggle/working/checkpoint_msmacro_hallu.csv
Empty/invalid hallucinations: 0
Saved hallucinated dataset: /kaggle/working/hallu_msmacro.csv
Saved labeled dataset: /kaggle/working/labeled_msmacro.csv
label
1    500
0    500
Name: count, dtype: int64


,id,question,context,answer,label
0,10736,buying a car dealer vs. private seller,[P1] Source: http://www.consumerhelp.ie/buying...,Traders are often called “dealers” and sell ca...,1
1,12719,convert dollar pay into salary to twice a month,[P1] Source: http://www.ehow.com/how_12044390_...,Determine what your paydays will be throughout...,1
2,17806,how to plant dwarf apple trees,[P1] Source: http://www.gardenguides.com/79766...,Locate a site for the dwarf apple tree where i...,1
3,14675,what arthritis is,[P1] Source: http://www.healthline.com/health/...,Arthritis is inflammation of the joints (the p...,1
4,16014,how to record audio with iphone,[P1] Source: http://www.solveyourtech.com/reco...,1: Open the Voice Memos app. If you cannot fin...,1


## 10. Final sanity checks

In [16]:
print("Gold dataset:", pd.read_csv(MSMARCO_500_PATH).shape)
print("Hallu dataset:", pd.read_csv(HALLU_PATH).shape)
print("Labeled dataset:", pd.read_csv(LABELED_PATH).shape)

display(pd.read_csv(MSMARCO_500_PATH).head(3))
display(pd.read_csv(HALLU_PATH).head(3))
display(pd.read_csv(LABELED_PATH).head(6))


Gold dataset: (500, 4)
Hallu dataset: (500, 5)
Labeled dataset: (1000, 5)


,id,question,context,gold_answer
0,10736,buying a car dealer vs. private seller,[P1] Source: http://www.consumerhelp.ie/buying...,Traders are often called “dealers” and sell ca...
1,12719,convert dollar pay into salary to twice a month,[P1] Source: http://www.ehow.com/how_12044390_...,Determine what your paydays will be throughout...
2,17806,how to plant dwarf apple trees,[P1] Source: http://www.gardenguides.com/79766...,Locate a site for the dwarf apple tree where i...


,id,question,context,gold_answer,hallu_answer
0,10736,buying a car dealer vs. private seller,[P1] Source: http://www.consumerhelp.ie/buying...,Traders are often called “dealers” and sell ca...,Buying from a dealer is always a better option...
1,12719,convert dollar pay into salary to twice a month,[P1] Source: http://www.ehow.com/how_12044390_...,Determine what your paydays will be throughout...,To convert your dollar pay into a salary that ...
2,17806,how to plant dwarf apple trees,[P1] Source: http://www.gardenguides.com/79766...,Locate a site for the dwarf apple tree where i...,"To plant dwarf apple trees, first, choose a lo..."


,id,question,context,answer,label
0,10736,buying a car dealer vs. private seller,[P1] Source: http://www.consumerhelp.ie/buying...,Traders are often called “dealers” and sell ca...,1
1,12719,convert dollar pay into salary to twice a month,[P1] Source: http://www.ehow.com/how_12044390_...,Determine what your paydays will be throughout...,1
2,17806,how to plant dwarf apple trees,[P1] Source: http://www.gardenguides.com/79766...,Locate a site for the dwarf apple tree where i...,1
3,14675,what arthritis is,[P1] Source: http://www.healthline.com/health/...,Arthritis is inflammation of the joints (the p...,1
4,16014,how to record audio with iphone,[P1] Source: http://www.solveyourtech.com/reco...,1: Open the Voice Memos app. If you cannot fin...,1
5,12260,how can the speed of light be used to measure ...,[P1] Source: http://www.answers.com/Q/How_can_...,As the speed of light is thought to be an abso...,1
